<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 20 · LLMs and Agents in the Asset Management Value Chain

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals

This notebook sketches simple agent-like patterns where a controller routes
high-level tasks to deterministic tools for diagnostics and reporting.


### Getting Help While Thinking About Agents

- Chapter 20 gives the conceptual view on agents and value chains.
- Appendix F discusses how to structure larger codebases around such agents
  and tools.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

import json

### Data Loading
Agents rely on the same price panel utilities as earlier notebooks.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=['Date']).set_index('Date').sort_index().ffill()

### Performance Helper

We reuse the standard performance statistics function so agents can ask for
quick diagnostics on strategies.


In [ ]:
def performance_stats(returns: pd.Series, risk_free: float = 0.02) -> pd.Series:
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - risk_free) / ann_vol if ann_vol > 0 else np.nan
    wealth = (1 + returns).cumprod()
    max_dd = (wealth / wealth.cummax() - 1).min()
    return pd.Series(
        {
            'annualized_return': ann_ret,
            'annualized_vol': ann_vol,
            'sharpe': sharpe,
            'max_drawdown': max_dd,
        }
    )

## 1. Tool Definitions

Agents interact with a small toolbox of deterministic functions: data loading,
constructing a simple strategy, and computing diagnostics.


In [ ]:
def load_prices() -> pd.DataFrame:
    return pd.read_csv(DATA_PATH,
        parse_dates=['Date']).set_index('Date').sort_index().ffill()


def simple_strategy(asset: str = 'AAPL') -> pd.Series:
    px = load_prices()[asset]
    rets = px.pct_change().dropna()
    signal = np.sign(rets.rolling(20).mean()).shift(1).dropna()
    strat_rets = rets.loc[signal.index] * signal
    return strat_rets


def strategy_report(asset: str = 'AAPL') -> dict:
    sr = simple_strategy(asset)
    stats = performance_stats(sr)
    out = {'asset': asset}
    out.update(stats.to_dict())
    return out

## 2. Minimal Agent Skeleton

Our agent chooses high-level tasks, invokes tools, and records a simple
history of what it did.


In [ ]:
class SimpleAgent:
    def __init__(self):
        self.history = []

    def run(self, task: str, **kwargs) -> dict:
        if task == 'diagnose_strategy':
            asset = kwargs.get('asset', 'AAPL')
            report = strategy_report(asset)
            self.history.append({'task': task, 'asset': asset, 'report': report})
            return report
        msg = {'error': f'Unknown task: {task}'}
        self.history.append(msg)
        return msg


agent = SimpleAgent()

## 3. Example: Strategy Health Check

We ask the agent to diagnose a simple momentum strategy on AAPL and
pretty-print the resulting report.


In [ ]:
from pprint import pprint

In [ ]:
report = agent.run('diagnose_strategy', asset='AAPL')
pprint(json.dumps(report, indent=2))

## 4. Exercises

### Exercise 1 – Drawdown Plot Tool
Add a tool that returns the drawdown series for a given strategy and plots it.
<details><summary>Hint</summary>Reuse the drawdown logic from Chapter 8 and
call it from the agent.</details>

### Exercise 2 – Multi-Asset Ranking Task
Teach the agent a new task that ranks a list of assets by recent Sharpe ratio.
<details><summary>Hint</summary>Loop over tickers, call simple_strategy and
performance_stats, then sort.</details>

### Exercise 3 – Chain-of-Thought Logging
Extend <code>self.history</code> to capture which tools were called and in
which order for each task.
<details><summary>Hint</summary>Store a list of step dictionaries with
timestamps and arguments.</details>


## 5. Takeaways

Even without a live LLM, you can design and exercise agent/task/tool
interfaces so that swapping in a foundation model later is mostly a
configuration change rather than a full rewrite.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">